# 01 · 今晚速成：一维卡尔曼滤波 + NIS

目标（约 1 小时）：
1. 搞懂卡尔曼滤波每一步在做什么（预测 → 更新）
2. 自己写出一维卡尔曼滤波（10 行代码）
3. 搞懂 **NIS** 是什么，用它判断滤波器的 Q、R 设得对不对

> 用法：从上往下 `Shift + Enter`。图里的标签用英文，避免中文字体显示成方块。

## 1. 一张图说清楚卡尔曼滤波

我们想估计一个看不见的真实值 $x$（比如体重），手上有两样东西：

- **模型**：我们对“它怎么变化”的认识，但模型不完美 → 误差方差 **Q**（过程噪声）
- **测量**：传感器读数 $z$，也不完美 → 误差方差 **R**（测量噪声）

卡尔曼滤波每一步做两件事，并且一直跟踪“我有多不确定”（方差 $P$）：

| 步骤 | 做什么 | 不确定性 |
|---|---|---|
| **预测** | 用模型往前推一步 | 变大：$\bar P = P + Q$ |
| **更新** | 拿到测量，按权重 $K$ 把估计往测量方向挪 | 变小：$P = (1-K)\bar P$ |

和第 1 章的 g-h 滤波对比：g-h 滤波里的 $g$ 是你**手动拍脑袋定的**；卡尔曼滤波里的 $K$ 是**每一步根据 $\bar P$ 和 $R$ 自动算出来的**。

## 2. 一维卡尔曼滤波的 5 个公式

模型：真实值每一步随机游走 $x_k = x_{k-1} + w_k$，$w_k \sim N(0, Q)$；测量 $z_k = x_k + v_k$，$v_k \sim N(0, R)$。

**预测**
$$\bar x = x, \qquad \bar P = P + Q$$

**更新**
$$
\begin{aligned}
y &= z - \bar x && \text{新息（innovation）：测量和预测差了多少}\\
S &= \bar P + R && \text{新息的方差：我“预期” } y \text{ 大概有多大}\\
K &= \bar P / S && \text{卡尔曼增益，0 到 1 之间}\\
x &= \bar x + K y\\
P &= (1-K)\,\bar P
\end{aligned}
$$

直觉：
- $\bar P \gg R$（我对预测很没把握、测量很准）→ $K \approx 1$ → 几乎直接相信测量
- $\bar P \ll R$（预测很有把握、测量很烂）→ $K \approx 0$ → 几乎不理测量

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2

def simulate(n, Q, R, x0=0.0, seed=0):
    # 生成“真实值”和“测量值”
    rng = np.random.default_rng(seed)
    xs = x0 + np.cumsum(rng.normal(0, np.sqrt(Q), size=n))   # 真实值：随机游走
    zs = xs + rng.normal(0, np.sqrt(R), size=n)              # 测量值：真实值 + 噪声
    return xs, zs

def kf_1d(zs, x0, P0, Q, R):
    # 一维卡尔曼滤波；返回每一步的估计、方差、增益、新息、新息方差
    x, P = x0, P0
    out = {"x": [], "P": [], "K": [], "y": [], "S": []}
    for z in zs:
        # --- 预测 ---
        x_bar = x
        P_bar = P + Q
        # --- 更新 ---
        y = z - x_bar
        S = P_bar + R
        K = P_bar / S
        x = x_bar + K * y
        P = (1 - K) * P_bar
        for k, v in zip(["x", "P", "K", "y", "S"], [x, P, K, y, S]):
            out[k].append(v)
    return {k: np.array(v) for k, v in out.items()}   # 把每个列表转成 numpy 数组

## 3. 跑起来看看

In [ ]:
Q_true, R_true = 0.05, 4.0
xs, zs = simulate(n=100, Q=Q_true, R=R_true, seed=1)
res = kf_1d(zs, x0=0.0, P0=10.0, Q=Q_true, R=R_true)

t = np.arange(len(zs))
sd = np.sqrt(res["P"])
plt.figure(figsize=(9, 4))
plt.scatter(t, zs, s=10, color="tab:red", alpha=0.5, label="measurement z")
plt.plot(t, xs, "k--", lw=1, label="truth x")
plt.plot(t, res["x"], color="tab:blue", label="KF estimate")
plt.fill_between(t, res["x"] - 2*sd, res["x"] + 2*sd, color="tab:blue", alpha=0.15, label="±2 sd")
plt.legend(); plt.xlabel("step"); plt.show()

In [ ]:
# 看看 K 和 P 怎么变化
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(res["K"]); ax[0].set_title("Kalman gain K")
ax[1].plot(res["P"]); ax[1].set_title("variance P")
plt.tight_layout(); plt.show()
print("最后的 K =", round(res["K"][-1], 3))

**观察**：一开始 $P_0=10$ 很大（对初值没把握）→ $K$ 接近 1，紧跟测量；几步之后 $K$ 和 $P$ 收敛到一个稳定值。
因为 $Q \ll R$（模型很稳、测量很吵），稳定后的 $K$ 很小，所以蓝线很平滑。

✏️ **动手**：把 `Q_true` 改成 `2.0` 再跑，$K$ 会变大还是变小？蓝线会怎样？

## 4. NIS：滤波器有没有“自欺欺人”？

现实中我们**不知道真实值**，那怎么判断 Q、R 设得对不对？只能看能观测到的东西——**新息 $y$**。

如果模型和 Q、R 都设对了，那么 $y_k \sim N(0, S_k)$：滤波器“预期”的新息大小 $S_k$ 和实际的新息 $y_k$ 应该匹配。
把它标准化：

$$\text{NIS}_k = \frac{y_k^2}{S_k} \qquad\Big(\text{多维时：}\ \varepsilon_k = y_k^\mathsf T S_k^{-1} y_k\Big)$$

- 一个 $N(0,1)$ 的平方服从 $\chi^2_1$，所以 **NIS 的均值应该 ≈ 1**（多维时 ≈ 测量维数 $m$）
- 对 $N$ 步取平均：$N\cdot\overline{\text{NIS}} \sim \chi^2_{N}$，于是可以算出 95% 的接受区间

怎么解读：

| 平均 NIS | 含义 |
|---|---|
| 明显 **> 区间上限** | 实际误差比滤波器以为的大 → 滤波器**过度自信**（Q 或 R 设小了，或模型错了） |
| 在区间内 | 一致（consistent），设定可信 |
| 明显 **< 区间下限** | 滤波器**过度保守**（Q 或 R 设大了） |

NIS 最大的好处：**不需要真实值**，真实数据上也能用。（第 8 章讲的 NEES 需要真实值，只能在仿真里用。）

In [ ]:
def nis_check(res, burn_in=10, alpha=0.05):
    nis = res["y"]**2 / res["S"]
    nis = nis[burn_in:]               # 去掉开头还没收敛的几步
    N = len(nis)
    lo, hi = chi2.ppf([alpha/2, 1 - alpha/2], df=N) / N
    mean = nis.mean()
    verdict = "OK" if lo <= mean <= hi else ("过度自信 (too big)" if mean > hi else "过度保守 (too small)")
    return mean, lo, hi, verdict

xs, zs = simulate(n=500, Q=Q_true, R=R_true, seed=2)
res = kf_1d(zs, 0.0, 10.0, Q_true, R_true)
mean, lo, hi, verdict = nis_check(res)
print(f"平均 NIS = {mean:.3f}，95% 区间 [{lo:.3f}, {hi:.3f}] -> {verdict}")

## 5. 实验：故意把 Q、R 设错

同一组数据，滤波器用不同的（错误的）Q、R，看 NIS 能不能“抓到”。

In [ ]:
cases = [
    ("correct",        Q_true,      R_true),
    ("R too small/10", Q_true,      R_true / 10),
    ("R too big x10",  Q_true,      R_true * 10),
    ("Q too small/50", Q_true / 50, R_true),
    ("Q too big x50",  Q_true * 50, R_true),
]
print(f"{'setting':<16}{'mean NIS':>10}   verdict")
for name, Qf, Rf in cases:
    r = kf_1d(zs, 0.0, 10.0, Qf, Rf)
    mean, lo, hi, verdict = nis_check(r)
    print(f"{name:<16}{mean:>10.3f}   {verdict}")
print(f"\n(95% 接受区间约为 [{lo:.3f}, {hi:.3f}])")

In [ ]:
# 画出 NIS 随时间的变化（滑动平均更好看）
def moving_avg(a, w=25):
    return np.convolve(a, np.ones(w)/w, mode="valid")

plt.figure(figsize=(9, 3.5))
for name, Qf, Rf in cases[:3]:
    r = kf_1d(zs, 0.0, 10.0, Qf, Rf)
    plt.plot(moving_avg(r["y"]**2 / r["S"]), label=name)
plt.axhline(1, color="k", ls="--", lw=1)
plt.yscale("log"); plt.ylabel("NIS (moving avg)"); plt.xlabel("step")
plt.legend(); plt.show()

**结论**：
- R 设小 → 滤波器以为测量很准，$S$ 偏小 → NIS 远大于 1
- R 设大 → $S$ 偏大 → NIS 小于 1
- Q 的影响类似，但通常不如 R 明显——NIS 只能告诉你“$S$ 整体偏大还是偏小”，**分不清是 Q 还是 R 的锅**。这是 NIS 的局限。

✏️ **思考题**：如果 Q 设小了、R 同时设大了，NIS 可能会怎样？这说明了什么？

## 6. 今晚的自检 ✅

- [ ] 预测这一步，为什么 $P$ 变大？更新这一步，为什么 $P$ 变小？
- [ ] $K$ 由什么决定？$K \to 1$ 和 $K \to 0$ 分别意味着什么？
- [ ] 新息 $y$ 和它的方差 $S$ 分别是什么？
- [ ] NIS 的定义是什么？为什么它的均值应该约等于 1？
- [ ] 平均 NIS 偏大说明什么？偏小呢？为什么 NIS 不需要真实值？

📝 **费曼复述**：用你自己的话，把“卡尔曼滤波在做什么”和“NIS 在检查什么”各讲一段，发给我。